<a href="https://colab.research.google.com/github/Talha-Shahid12/LLM-FOR-AMBULANCE/blob/main/Conversions_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/openai/whisper.git

In [ ]:
!pip install --upgrade --no-deps --force-reinstall git+https://github.com/openai/whisper.git

In [ ]:
!sudo apt update && sudo apt install ffmpeg

In [ ]:
!pip install fastapi nest-asyncio pyngrok uvicorn

In [ ]:
!pip install python-multipart

In [ ]:
!pip install googletrans==4.0.0-rc1

In [ ]:
!pip install gtts

In [ ]:
!pip install playsound


In [ ]:
!pip install pygobject

In [ ]:
!pip install pydub numpy torch

In [50]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [43]:
from googletrans import Translator
from gtts import gTTS
import playsound
from google.colab import userdata
from fastapi.responses import FileResponse
from fastapi import FastAPI, UploadFile, Form
import whisper
import nest_asyncio
import uvicorn
import numpy as np
import torch
import os
import io
from pydub import AudioSegment
import nest_asyncio
from pyngrok import ngrok
import uvicorn

In [ ]:
model = whisper.load_model("base")

In [53]:
#set ngork auth token in colab enviorment
authtoken=userdata.get('authtoken')

In [39]:
!ngrok authtoken authtoken

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [27]:
class TTS_STT_Service:
    def __init__(self):
        self.translator = Translator()

    def text_to_speech(self, text, output_file="output.mp3"):
        """
        Translates text to Urdu and converts it to speech.
        """
        # Translate to Urdu
        translated_text = self.translator.translate(text, src='en', dest='ur').text
        print(f"Translated Text: {translated_text}")

        # Convert to speech
        tts = gTTS(text=translated_text, lang='ur')
        tts.save(output_file)

        # Play the audio
        playsound.playsound(output_file)
        return output_file

    def speech_to_text(self, file_path, task="translate"):
        """
        Transcribes or translates speech from an audio file.
        """
        result = model.transcribe(file_path, task=task)
        return result["text"]

In [51]:
class TTS_STT_Controller:
    def __init__(self):
        self.service = TTS_STT_Service()

    @app.post("/text-to-voice")
    async def text_to_voice(text: str = Form(...), language: str = Form("ur")):
      try:
          # Translate text
          translator = Translator()
          translated_text = translator.translate(text, src="en", dest=language).text

          # Convert to speech
          output_file = "/content/drive/MyDrive/voices/output.mp3"
          tts = gTTS(text=translated_text, lang=language)
          tts.save(output_file)

          # Return file URL or confirmation
          return {"message": "Audio generated successfully", "audio_file": output_file}
      except Exception as e:
        return {"error": str(e)}

    async def handle_voice_to_text(self, file: UploadFile):
        """
        Handle voice-to-text transcription endpoint for directly uploaded files.
        """
        try:
            # Read the file content directly
            audio_data = await file.read()

            # Save temporarily to memory or process it directly
            temp_file_path = f"./{file.filename}"
            with open(temp_file_path, "wb") as temp_file:
                temp_file.write(audio_data)

            # Transcribe the file
            transcription = self.service.speech_to_text(temp_file_path)

            # Clean up temporary file
            os.remove(temp_file_path)
            return {"transcription": transcription}

        except Exception as e:
            return {"error": str(e)}

In [32]:
controller = TTS_STT_Controller()

In [33]:
app = FastAPI()

In [48]:
@app.post("/text-to-voice")
async def text_to_voice(text: str = Form(...)):
    """
    API endpoint to convert English text to Urdu speech.
    """
    return await controller.handle_text_to_voice(text)


@app.post("/voice-to-text")
async def voice_to_text(file: UploadFile):
    """
    API endpoint to handle voice-to-text transcription for uploaded voice files.
    """
    return await controller.handle_voice_to_text(file)


@app.get("/download-audio")
async def download_audio():
    file_path = "/content/drive/MyDrive/voices/output.mp3"
    return FileResponse(path=file_path, media_type="audio/mpeg", filename="output.mp3")

In [35]:
PORT=8000

In [ ]:
ngrok_tunnel = ngrok.connect(PORT)
print(f"Public URL: {ngrok_tunnel.public_url}")

nest_asyncio.apply()
uvicorn.run(app, port=PORT)

In [ ]:
#For Testing purpose direct in colab envoirment
!whisper "/content/sample.opus" --model medium --task translate